In [ ]:
import pandas as pd
import pandas as pd
import utils
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold, ParameterGrid
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, TargetEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer, make_column_selector
import importlib

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = raw_train_df.columns[raw_train_df.columns.str.startswith('tof')]
non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
exp_name = 'imu_only_simple'
exp_name = 'fft_v1_baseline' # Named for your experiment
sampling_rate = 100
eg_subject = 'SUBJ_040724'
single_sequence = 'SEQ_065189'
dc_offset_max = 10
phase_1_cols = ['acc_x', 'acc_y', 'acc_z', 'sequence_counter', 'phase', 'behavior', 'orientation', 'subject', 'gesture', 'sequence_id']
log_edges = np.logspace(np.log10(0.5), np.log10(100), num=15)
sequences = ['SEQ_000034', 'SEQ_065478', 'SEQ_065470']
subjects = ['SUBJ_059520']
band_edges = np.arange(0, 101, 10)
pipe_name = 'imu_extractor'
categorical_features = ['orientation', 'subject']
accelerometer_combinations = [
    ['acc_x'], ['acc_y'], ['acc_z'],
    ['acc_x', 'acc_y'], ['acc_x', 'acc_z'], ['acc_y', 'acc_z'],
    ['acc_x', 'acc_y', 'acc_z']
]
subject_cols = ['orientation', 'subject', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000016', 'SEQ_000018',
                    'SEQ_000022', 'SEQ_000033', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053',
                    'SEQ_000058', 'SEQ_000063', 'SEQ_000079', 'SEQ_000091', 'SEQ_000092',
                    'SEQ_000111', 'SEQ_000113', 'SEQ_000114', 'SEQ_000142', 'SEQ_000150']
some_subjects = ['SUBJ_059520', 'SUBJ_020948', 'SUBJ_040282', 'SUBJ_052342', 'SUBJ_032165',
                'SUBJ_024086', 'SUBJ_040733', 'SUBJ_063346', 'SUBJ_055211', 'SUBJ_001430',
                'SUBJ_012088', 'SUBJ_040310', 'SUBJ_032233', 'SUBJ_059330', 'SUBJ_013623',
                'SUBJ_032585', 'SUBJ_063464', 'SUBJ_038023', 'SUBJ_044680', 'SUBJ_024137']

In [ ]:
train_df = raw_train_df.set_index('row_id')

single_subject_df = train_df.loc[train_df['subject'] == eg_subject, :]

single_gesture_df = single_subject_df.loc[single_subject_df['sequence_id'] == single_sequence]

multiple_gesture_df = train_df.loc[train_df['sequence_id'].isin(some_sequences), phase_1_cols]

some_subjects_df = train_df.loc[train_df['subject'].isin(some_subjects), phase_1_cols]

In [ ]:
importlib.reload(utils)

num_pattern = 'acc'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols = ['orientation', 'subject']
normal_cols = ['adult_child', 'sex', 'handedness', 'segment_id']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            OrdinalEncoder(),
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

pipeline = Pipeline([
    (pipe_name, utils.ImuExtractor(subject_df=train_demo_df,
                                   imu_sensor_list=['acc_x'],
                                   )),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(RandomForestClassifier()))
])

param_grid = {
    f'{pipe_name}__imu_sensor_list': [['acc_x']],
    f'{pipe_name}__domain': ['acceleration'],
    f'{pipe_name}__dc_offset': [2.0],
    f'{pipe_name}__band_edges': [None],
    f'{pipe_name}__sampling_rate': [100],
    f'{pipe_name}__category_data': [True]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=3),
    verbose=1,
    n_jobs=1
)

y = multiple_gesture_df[['sequence_id', 'gesture']]

grid_search.fit(multiple_gesture_df, y, groups=multiple_gesture_df['sequence_id'])

pd.DataFrame(grid_search.cv_results_)

In [ ]:
importlib.reload(utils)

pipeline = Pipeline([
    (pipe_name, utils.ImuExtractor(
                                   imu_sensor_list=['acc_x'],
                                   category_data=True,
                                   segmentation='both'
                                   )),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(RandomForestClassifier()))
])

pipeline.named_steps['imu_extractor'].transform(single_gesture_df)

In [ ]:
train_df.groupby('sequence_id')['sequence_counter'].max().plot(kind='box')

In [ ]:
split_by = 'behavior'
rah = pipeline.named_steps['imu_extractor'].split_segments(single_gesture_df, split_by)

singular_record_list = []
for a_segment in rah:
    imu_df = pipeline.named_steps['imu_extractor'].process_for_imu_values(a_segment)
    imu_features_df = pipeline.named_steps['imu_extractor'].extract_features_from_imu(imu_df, None)
    category_df = pipeline.named_steps['imu_extractor'].return_single_category_desc_record(a_segment, split_by)
    singular_record_list.append(pd.concat([imu_features_df, category_df], axis=1))
    
res = pd.concat(singular_record_list, axis=0)
res